# D - Dependency Inversion (Inversión de Dependencias)
Ejemplo: gestión de puntajes altos (HighScore). Violación: depender de implementación concreta (FileStorage). Corrección: depender de abstracción de almacenamiento e inyectarla.


In [ ]:
# --- Viola DIP: HighScoreManager depende directamente de FileStorage ---
class FileStorage:
    def __init__(self, path: str):
        self.path = path
        self._last_written = None
    def write(self, data):
        import json
        with open(self.path, 'w') as f:
            json.dump(data, f)
        self._last_written = data
    def read(self):
        import json
        with open(self.path, 'r') as f:
            return json.load(f)

class HighScoreManagerViolation:
    def __init__(self, storage_path: str):
        self.storage = FileStorage(storage_path)
        self.scores = []
    def add_score(self, name: str, points: int):
        self.scores.append({'name': name, 'points': points})
    def save(self):
        self.storage.write(self.scores)

hsv = HighScoreManagerViolation('scores_violation.json')
hsv.add_score('A', 10)
hsv.save()
# Si quiero cambiar a DB, tengo que modificar HighScoreManagerViolation -> dependemos de concreción


In [ ]:
# --- Corrección: definir abstracción Storage y inyectar implementación ---
from typing import Protocol, List

class Storage(Protocol):
    def write(self, data): ...
    def read(self): ...

class FileStorageFixed:
    def __init__(self, path: str):
        self.path = path
    def write(self, data):
        import json
        with open(self.path, 'w') as f:
            json.dump(data, f)
    def read(self):
        import json, os
        if not os.path.exists(self.path):
            return []
        with open(self.path, 'r') as f:
            return json.load(f)

class InMemoryStorage:
    def __init__(self):
        self._data = []
    def write(self, data):
        self._data = data
    def read(self):
        return self._data

class HighScoreManagerFixed:
    def __init__(self, storage: Storage):
        self.storage = storage
        self.scores: List[dict] = []
    def add_score(self, name: str, points: int):
        self.scores.append({'name': name, 'points': points})
    def save(self):
        self.storage.write(self.scores)
    def load(self):
        self.scores = self.storage.read()

# uso con in-memory (útil para tests) y con file (producción)
mem = InMemoryStorage()
hmf = HighScoreManagerFixed(mem)
hmf.add_score('Player1', 50)
hmf.save()
print('In-memory saved:', mem.read())
# cambiar a FileStorageFixed sin modificar HighScoreManagerFixed
fs = FileStorageFixed('scores_fixed.json')
hmf2 = HighScoreManagerFixed(fs)
hmf2.add_score('Player2', 80)
hmf2.save()
print('File saved ok')
